# CranioVision — Clinical Report Generator Demo

**Phase 3 Week 3 deliverable.**

End-to-end: takes one case, runs the full pipeline, generates a 4-page clinical PDF.

**Pages produced:**
1. Clinical Summary — hero figure of chosen prediction + key findings
2. Multi-model Comparison — per-model metrics + side-by-side thumbnails
3. Anatomical Context — lobe distribution + eloquent cortex risk
4. Feature Attention — Grad-CAM heatmaps with methodology

**Runtime:** ~6-8 min (full pipeline) + ~5 sec (PDF assembly)

**Requires:** all 3 trained checkpoints, ANTs registration cache (built in Week 1), reportlab installed.

## 1. Setup

In [1]:
import sys
import time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import torch

from src.cranovision.config import OUTPUTS_DIR
from src.cranovision.data import get_splits, get_val_transforms
from src.cranovision.pipeline import (
    run_full_analysis,
    compute_xai_for_model,
)
from src.cranovision.reporting import generate_clinical_report

print('Imports ok')

c:\Users\hrana\anaconda3\envs\ml_env_fixed\Lib\site-packages\ignite\handlers\checkpoint.py:17: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


Imports ok


In [2]:
# Configuration
CASE_IDX = 0

# Which prediction headlines page 1 + page 3.
# Options: 'attention_unet' | 'swin_unetr' | 'nnunet' | 'ensemble'
PREDICTION_TO_FEATURE = 'ensemble'

# Pick the case
_, _, test_cases = get_splits()
case = test_cases[CASE_IDX]
CASE_ID = case['case_id']
print(f'Case: {CASE_ID}')
print(f'Featured prediction: {PREDICTION_TO_FEATURE}')

Scanning 200 patient folders in BraTS2024_small_dataset/
  Valid cases: 200
Loading existing split from data_split.json
  Train: 140 (70%) | Val: 30 (15%) | Test: 30 (15%)
Case: BraTS-GLI-02196-105
Featured prediction: ensemble


## 2. Run full analysis pipeline

Runs all 3 models + ensemble + atlas + per-model anatomy & eloquent. ~6-8 min.

In [3]:
def progress(stage, pct, msg):
    print(f'  [{pct:3d}%] {stage:<14}: {msg}')

t0 = time.time()
analysis = run_full_analysis(
    case_dict=case,
    progress_fn=progress,
    include_atlas=True,
)
elapsed = time.time() - t0
print(f'\nFull analysis complete in {elapsed/60:.1f} min')

  [  0%] init          : Starting analysis for BraTS-GLI-02196-105
  [  5%] preprocess    : Loading and normalizing MRI volumes


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


  [ 10%] inference     : Running Attention U-Net...
Loading attention_unet from attention_unet_best.pth...
  ✓ Loaded. Device: cuda


`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


  [ 26%] inference     : Running SwinUNETR...
Loading swin_unetr from swin_unetr_best.pth...
  ✓ Loaded. Device: cuda
  [ 43%] inference     : Running nnU-Net DynUNet...
Loading nnunet from nnunet_best.pth...
  ✓ Loaded. Device: cuda
  [ 65%] ensemble      : Computing weighted ensemble
  [ 70%] metrics       : Computing volumes and metrics
  [ 75%] agreement     : Computing model agreement
  [ 80%] atlas         : Registering patient T1 to MNI152
  [ 92%] atlas         : Computing anatomical context
  [100%] done          : Analysis complete

Full analysis complete in 2.5 min


In [4]:
# Quick sanity check on the analysis output
print('Predictions returned:', list(analysis['predictions'].keys()))
print('Available models    :', analysis['available_models'])
print('Agreement           :', analysis['agreement'])

for name, m in analysis['per_model_metrics'].items():
    vol_total = m['volumes_cm3'].get('Total tumor', 0)
    dice = m.get('mean_dice', 'n/a')
    if isinstance(dice, float):
        dice = f'{dice:.4f}'
    print(f'  {name:<18}: total={vol_total:6.2f} cm^3, mean Dice={dice}')

# Check if atlas worked
atlas = analysis.get('atlas', {})
if '_error' in atlas:
    print(f'\nAtlas global error: {atlas["_error"]}')
else:
    for name in analysis['predictions']:
        if name in atlas:
            sub = atlas[name]
            if 'error' in sub:
                print(f'  atlas {name}: ERROR {sub["error"]}')
            else:
                primary = sub.get('anatomy', {}).get('primary_region', '?')
                print(f'  atlas {name:<18}: primary={primary}')

Predictions returned: ['attention_unet', 'swin_unetr', 'nnunet', 'ensemble']
Available models    : ['attention_unet', 'swin_unetr', 'nnunet']
Agreement           : {'unanimous_fraction': 0.9841, 'n_models_compared': 3}
  attention_unet    : total=175.31 cm^3, mean Dice=0.8201
  swin_unetr        : total=213.35 cm^3, mean Dice=0.9067
  nnunet            : total=245.89 cm^3, mean Dice=0.8862
  ensemble          : total=212.41 cm^3, mean Dice=0.9117
  atlas attention_unet    : primary=Frontal Lobe
  atlas swin_unetr        : primary=Frontal Lobe
  atlas nnunet            : primary=Frontal Lobe
  atlas ensemble          : primary=Frontal Lobe


## 3. Compute XAI for the chosen prediction

This loads Attention U-Net (the shared explainer) and computes Grad-CAM heatmaps for all 3 tumor classes. ~30 sec on GPU.

In [5]:
t0 = time.time()
xai = compute_xai_for_model(
    model_name=PREDICTION_TO_FEATURE,    # which prediction to explain
    case_dict=case,
    progress_fn=progress,
)
elapsed = time.time() - t0
print(f'\nXAI complete in {elapsed:.1f}s')

print(f'Explainer used         : {xai["explainer_model"]}')
print(f'Prediction explained   : {xai["prediction_being_explained"]}')
print(f'Heatmap shapes         : {[hm.shape for hm in xai["heatmaps"].values()]}')
print(f'Device used            : {xai.get("device_used", "?")}')

  [  5%] xai_init      : Loading explainer (Attention U-Net)
Loading attention_unet from attention_unet_best.pth...
  ✓ Loaded. Device: cuda
  [ 30%] xai_compute   : Running patch-based Grad-CAM


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


  [100%] xai_done      : Explanation ready

XAI complete in 38.3s
Explainer used         : attention_unet
Prediction explained   : ensemble
Heatmap shapes         : [torch.Size([160, 192, 160]), torch.Size([160, 192, 160]), torch.Size([160, 192, 160])]
Device used            : cuda:0


## 4. Generate the PDF report

In [6]:
# We need the preprocessed image for figures, but the analysis dict
# doesn't return it (kept JSON-friendly). Recompute it locally.
# (In production, this comes from preprocessing the upload.)
tx = get_val_transforms()
sample = tx(case)
preprocessed_image = sample['image']
print(f'Preprocessed image shape: {preprocessed_image.shape}')

t0 = time.time()
pdf_path = generate_clinical_report(
    case_id=CASE_ID,
    analysis_result=analysis,
    xai_result=xai,
    prediction_to_feature=PREDICTION_TO_FEATURE,
    image=preprocessed_image,
)
elapsed = time.time() - t0
print(f'\nPDF generated in {elapsed:.1f}s')
print(f'Saved: {pdf_path}')
print(f'Size : {pdf_path.stat().st_size / 1024:.1f} KB')

Preprocessed image shape: torch.Size([4, 160, 192, 160])

PDF generated in 2.0s
Saved: D:\2_ML PROJECTS\30. Brainstorm\CranioVision\outputs\reports\BraTS-GLI-02196-105_clinical_report.pdf
Size : 1126.5 KB


## 5. Open the PDF

Cross-platform open. On Windows this launches the default PDF viewer.

In [7]:
import os
import platform
import subprocess

if platform.system() == 'Windows':
    os.startfile(str(pdf_path))
elif platform.system() == 'Darwin':
    subprocess.run(['open', str(pdf_path)])
else:
    subprocess.run(['xdg-open', str(pdf_path)])
print('Opening PDF in default viewer...')

Opening PDF in default viewer...


## 6. (Optional) Generate reports for the other 3 predictions

Demonstrates that the radiologist's UI can produce a different-headlined report depending on which prediction they trust. Each PDF is identical except for page 1 and 3 which highlight the chosen model.

In [8]:
# Skip this cell if you only want the one report — comment out the body.
RUN_ALL_VARIANTS = False

if RUN_ALL_VARIANTS:
    other_predictions = ['attention_unet', 'swin_unetr', 'nnunet']
    other_predictions = [p for p in other_predictions
                         if p != PREDICTION_TO_FEATURE
                         and p in analysis['predictions']]

    for pred_name in other_predictions:
        out = OUTPUTS_DIR / 'reports' / f'{CASE_ID}_clinical_report_{pred_name}.pdf'
        path = generate_clinical_report(
            case_id=CASE_ID,
            analysis_result=analysis,
            xai_result=xai,                       # XAI is reused — same explainer
            prediction_to_feature=pred_name,
            image=preprocessed_image,
            output_path=out,
        )
        print(f'  {pred_name:<18}: {path.name}')
    print('\nAll variants generated')
else:
    print('Set RUN_ALL_VARIANTS = True to generate the other 3 variants.')

Set RUN_ALL_VARIANTS = True to generate the other 3 variants.


## What just happened

1. **Full pipeline executed.** All 3 models, ensemble, anatomy, eloquent — runs once on the case.
2. **XAI computed for the chosen prediction.** Attention U-Net acts as the shared explainer (architectural decision validated in Week 2).
3. **4-page PDF assembled.** Page 1 features the chosen model's segmentation. Page 2 compares all 4 predictions. Page 3 shows anatomical context for the chosen prediction. Page 4 shows feature attention heatmaps.
4. **PDF written + auto-opened.**

**This closes Phase 3.** The full clinical pipeline is now end-to-end:  
`MRI -> 4 predictions -> anatomy -> eloquent risk -> XAI -> PDF report`

**Next: Phase 4 — building the FastAPI backend + v0.dev frontend that calls this same pipeline.**